In [10]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import rateslib as rl
import QuantLib as ql

from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB
from Query.FixedRateBonds.FixedRateBondQuery import FixedRateBondQuery 
from Query.FixedRateBonds.FixedRateBondStructure import FixedRateBondStructure, FixedRateBondStructureFunctionMap 
from Query.FixedRateBonds.FixedRateBondValue import FixedRateBondValue, FixedRateBondValueFunctionMap 

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue

from TB.TimeseriesBuilder import TimeseriesBuilder

# fmt: off
import Query.FixedRateBonds.adapter  # noqa: F401
# fmt: on

from utils.ql_utils import datetime_to_ql_date, ql_date_to_datetime 

In [12]:
frb_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-QL")
frb_tb = FixedRateBondsTB(mdp=frb_mdp)

In [17]:
start = datetime.date(2025, 10, 8)
end = datetime.date(2025, 10, 8)

with frb_mdp:
    df = frb_tb.get_timeseries(
        start=start,
        end=end,
        queries=[
            # FixedRateBondQuery(cusip="oo2/oo5/oo10", value=FixedRateBondValue.YTM),
            FixedRateBondQuery(cusip="ct2/ct5/ct10", value=FixedRateBondValue.YTM),
            FixedRateBondQuery(cusip="ct2/ct5/ct10", value=FixedRateBondValue.YTM, structure_kwargs={"risk_weights": [0.60, 1, -0.32]}),
            FixedRateBondQuery(cusip="ct2/ct5/ct10", value=FixedRateBondValue.YTM, structure_kwargs={"risk_weights": [-0.50, 1, -0.50]}),
        ],
        ignore_cache=True,
        n_jobs=1,
    )

PRICING FIXED-RATE BONDS.: 100%|██████████| 3/3 [00:01<00:00,  2.54it/s]


In [18]:
df
# plt.plot(df)

,ct2/ct5/ct10 FLY YTM,ct2/ct5/ct10 0.6/1/-0.32 FLY YTM,ct2/ct5/ct10 FLY YTM
Date,,,
2025-10-08,-27.7,455.5,-13.85


In [13]:
FixedRateBondQuery(cusip="ct2/ct5/ct10", value=FixedRateBondValue.YTM, structure_kwargs={"risk_weights": [0.60, -1, 0.32]}).col_name()

'ct2/ct5/ct10 0.6/-1/0.32 FLY YTM'

In [8]:
q = FixedRateBondQuery(cusip="o10/ct10", value=FixedRateBondValue.YTM, structure=FixedRateBondStructure.CURVE)
q.col_name()

'o10/ct10 CURVE YTM'

In [6]:
frb_mdp._get_single_pricer(cusip="0540/20", timestamp="live")

QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2025, 10, 8), _issue_date=datetime.date(2020, 6, 1), _maturity_date=datetime.date(2040, 5, 15), _cpn=1.125, _notional=None, _clean_price=None, _ytm=np.float64(4.538), _meta_data={'record_date': datetime.date(2020, 6, 1), 'label': 'T 1 1/8 May 40', 'cusip': '912810SR0', 'oi': '20-Year', 'auction_date': datetime.date(2020, 5, 20), 'issue_date': datetime.date(2020, 6, 1), 'maturity_date': datetime.date(2040, 5, 15), 'cpn': 1.125, 'rank': 21, 'timestamp': Timestamp('2025-10-08 21:00:36+0000', tz='UTC')})

In [29]:
ust_pricers = frb_mdp._get_multi_pricers(cusips=["ct2/ct10"], timestamp="live")
ust_pricers

{'ct2': QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2025, 10, 8), _issue_date=datetime.date(2025, 9, 30), _maturity_date=datetime.date(2027, 9, 30), _cpn=3.5, _notional=None, _clean_price=None, _ytm=np.float64(3.572), _meta_data={'record_date': datetime.date(2025, 9, 30), 'label': 'T 3 1/2 Sep 27', 'cusip': '91282CPB1', 'oi': '2-Year', 'auction_date': datetime.date(2025, 9, 23), 'issue_date': datetime.date(2025, 9, 30), 'maturity_date': datetime.date(2027, 9, 30), 'cpn': 3.5, 'rank': 0, 'timestamp': Timestamp('2025-10-08 11:43:39+0000', tz='UTC')}),
 'ct10': QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2025, 10, 8), _issue_date=datetime.date(2025, 8, 15), _maturity_date=datetime.date(2035, 8, 15), _cpn=4.25, _notional=None, _clean_price=None, _ytm=np.float64(4.105), _meta_data={'record_date': datetime.date(2025, 8, 15), 'label': 'T 4 1/4 Aug 35', 'cusip': '91282CNT4', 'oi': '10-Year', 'auction_date': datetime.date(2025, 8, 6), 'issue

In [30]:
tenor = "ct5/ct10/ct30", FixedRateBondStructure.FLY 
timestamp = "live"

query = FixedRateBondQuery(cusip=tenor[0], structure=tenor[1], structure_kwargs={"bpv": 10_000})
pricer = frb_mdp._get_multi_pricers(cusips=[query.cusip], timestamp=timestamp)
pkg, rws = query.resolve_package(pricer_or_curve=pricer)
vmap = query.build_value_map(pricer_or_curve=pricer, package=pkg, risk_weights=rws)

In [31]:
pkg, rws

([<QuantLib.QuantLib.FixedRateBond; proxy of <Swig Object of type 'ext::shared_ptr< FixedRateBond > *' at 0x0000021FE5BD3690> >,
  <QuantLib.QuantLib.FixedRateBond; proxy of <Swig Object of type 'ext::shared_ptr< FixedRateBond > *' at 0x0000021FE4596100> >,
  <QuantLib.QuantLib.FixedRateBond; proxy of <Swig Object of type 'ext::shared_ptr< FixedRateBond > *' at 0x0000021FE4597DB0> >],
 [np.float64(-1.0), np.float64(2.0), np.float64(-1.0)])

In [32]:
# vmap.apply(FixedRateBondValue.NPV)
vmap.apply(FixedRateBondValue.YTM)
# vmap.apply(FixedRateBondValue.CLEAN_PRICE)

np.float64(-18.50000000000005)

In [7]:
frb_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-QL")

In [10]:
frb_pricer = frb_mdp._get_single_pricer(cusip="ct2", timestamp=datetime.date(2010, 9, 29))
frb_pricer

QLFixedRateBondPricer(_ql_frb_id='USTS', _reference_date=datetime.date(2010, 9, 29), _issue_date=datetime.date(2010, 8, 31), _maturity_date=datetime.date(2012, 8, 31), _cpn=0.375, _notional=None, _clean_price=99.90625, _ytm=None, _meta_data={'record_date': '2010-08-31', 'label': 'T 0 3/8 Aug 12', 'cusip': '912828PH7', 'oi': '2-Year', 'auction_date': '2010-08-24', 'issue_date': '2010-08-31', 'maturity_date': '2012-08-31', 'cpn': 0.375, 'rank': 0})

In [ ]:
# from MDP.FixedRateBonds.FEDINVEST.FedInvestFetcher import FedInvestDataFetcher

# timestamp_dt = datetime.datetime(2025, 10, 7)

# FedInvestDataFetcher().runner(dates=[timestamp_dt])

C:\Users\chris\AppData\Local\ARBS\Cache\zodb\dump\FedInvest_Prices_Cache.fs


{datetime.datetime(2025, 10, 7, 0, 0):          cusip               type  coupon  offer_price  bid_price   eod_price
 0    912797QE0  MARKET BASED BILL   0.000     0.000000  99.977667   99.988833
 1    912797RC3  MARKET BASED BILL   0.000    99.921931  99.921639   99.932833
 2    912797QF7  MARKET BASED BILL   0.000    99.899875  99.899500   99.910667
 3    912797RD1  MARKET BASED BILL   0.000    99.843861  99.843278   99.854472
 4    912797QG5  MARKET BASED BILL   0.000    99.821333  99.820889   99.832500
 ..         ...                ...     ...          ...        ...         ...
 454  91282CLA7   MARKET BASED FRN   4.126   100.005537  99.993456   99.989430
 455  91282CLT6   MARKET BASED FRN   4.150   100.024117  99.992612  100.010985
 456  91282CMJ7   MARKET BASED FRN   4.041    99.892022  99.869424   99.875879
 457  91282CMX6   MARKET BASED FRN   4.104    99.956004  99.933151   99.933151
 458  91282CNQ0   MARKET BASED FRN   4.103    99.949919  99.934513   99.934513
 
 [459 rows x